In [13]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q polars
import polars as pl

BASE = "/content/drive/MyDrive/Capstone"

path_meta = f"{BASE}/combined_galveston_metadata_reduced.parquet"
path_ts   = f"{BASE}/combined_galveston_reduced.parquet"
path_out  = f"{BASE}/combined_galveston_mapped.parquet"

# Load metadata once (small file)
meta = (
    pl.read_parquet(path_meta)
      .select(["meter_uuid", "meter_size_inches", "h3_index_res12"])
      .unique(subset=["meter_uuid"])
)

# Lazy scan large time-series, join, and write entire output to parquet
(
    pl.scan_parquet(path_ts)
      .with_columns(pl.col("meter_uuid").cast(pl.Utf8))
      .join(
          meta.lazy().with_columns(pl.col("meter_uuid").cast(pl.Utf8)),
          on="meter_uuid",
          how="left"
      )
      .sink_parquet(path_out)
)

print("Merge Successful!")
print(f"Saved to: {path_out}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Merge Successful
Saved to: /content/drive/MyDrive/Capstone/combined_galveston_mapped.parquet
